# 신장생리 실습 (1시간 축약판)

**진행 방식**
1. 케이스와 데이터를 읽는다.
2. **실행하기 전에** 🤔 생각해 보기 질문에 대해 결과가 어떻게 될지 머릿속으로 예상해 본다. 적을 필요는 없다.
3. ▶ 셀을 실행하고 슬라이더를 움직여 확인한다.
4. 예상과 다른 결과가 나왔다면 왜 그런지 생각해 본다. 틀린 예상에서 가장 많이 배운다.
5. 마지막에 **스크린샷과 짧은 감상**을 pptx 파일 하나로 만들어 PLATO에 제출한다. (맨 아래 제출 방법 참고)

> 코드를 몰라도 됩니다. 위에서부터 차례로 ▶ 셀을 실행하고 슬라이더만 움직이세요.
> 모든 모델은 교육용으로 단순화한 것입니다.

| 모듈 | 주제 | 시간 |
|---|---|---|
| 준비 | Colab 사용법, 준비 셀 실행 | 5분 |
| M1 | Clearance와 glucose titration | 20분 |
| M2 | 사구체 Starling 역학 | 20분 |
| M3 | 소변 농축·희석의 정량화 | 15분 |

### 🖥️ Colab 사용법 (처음이라면 꼭 읽기)
0. **가장 먼저 메뉴의 `파일 → Drive에 사본 저장`을 누르세요.** 새 탭에 *"…의 사본"*이 열리면 **그 사본에서** 실습합니다. (사본을 만들지 않으면 작업 내용이 저장되지 않습니다.)
1. 회색 코드 셀은 **제목만 보이고 코드는 숨겨져** 있습니다. 셀 왼쪽의 ▶ 버튼을 누르거나, 셀을 클릭한 뒤 `Shift+Enter`를 누르면 실행됩니다.
2. **맨 위 ⚙️ 준비 셀을 먼저 실행**하세요. 실행하지 않으면 아래 셀에서 `NameError`가 납니다.
3. 오류가 나거나 화면이 꼬이면 메뉴에서 **런타임 → 모두 실행**을 누르면 처음부터 다시 실행됩니다.
4. 글자만 있는 텍스트 셀은 읽기만 하면 됩니다. 노트북 안에 직접 쓸 곳은 없습니다.
5. **Google 계정으로 로그인한 상태**에서 진행하세요. 제출 스크린샷에 내 계정이 보여야 합니다.

In [ ]:
#@title ⚙️ 준비: 이 셀을 가장 먼저 실행하세요 (코드는 몰라도 됩니다)
# 이 셀은 실습에 필요한 계산·그래프 함수를 불러옵니다. 내용을 읽을 필요는 없습니다.
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

try:  # Colab에서 슬라이더가 잘 보이도록
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})


def clearance(U, P, V):
    """C = U*V/P (U and P in the same units; V in mL/min)."""
    return U * V / P


def check_m1_1(V=1.2, Hct=0.45):
    gfr = clearance(55, 0.5, V)
    rpf = clearance(10, 0.02, V)
    ccr = clearance(115, 1.0, V)
    na_filt = gfr * 140 / 1000  # mEq/min
    na_excr = 40 * V / 1000  # mEq/min
    print(f"GFR (inulin)      = {gfr:.0f} mL/min")
    print(f"RPF (PAH, 'ERPF') = {rpf:.0f} mL/min  -> RBF = {rpf/(1-Hct):.0f} mL/min")
    print(f"FF                = {gfr/rpf:.2f}")
    print(f"C_cr              = {ccr:.0f} mL/min  ({100*(ccr/gfr-1):+.0f}% vs inulin)")
    print(
        f"Na filtered load  = {na_filt:.1f} mEq/min ; excreted = {na_excr:.3f} mEq/min"
    )
    print(f"FE_Na             = {100*na_excr/na_filt:.2f} %")


cases_aki = {
    "A: 구토 3일, BP 92/60": dict(
        P_Na=138, U_Na=10, P_cr=2.5, U_cr=150, P_bun=60, U_urea=900
    ),
    "B: 패혈성 쇼크 후 4일째": dict(
        P_Na=138, U_Na=60, P_cr=3.0, U_cr=30, P_bun=55, U_urea=250
    ),
    "C: 심부전, furosemide 복용 중": dict(
        P_Na=136, U_Na=70, P_cr=2.4, U_cr=100, P_bun=60, U_urea=700
    ),
}


def fe(U_x, P_x, U_cr, P_cr):
    """Fractional excretion (%) = (U_x * P_cr) / (P_x * U_cr) * 100. Independent of V."""
    return 100 * U_x * P_cr / (P_x * U_cr)


def show_aki(reveal=False):
    for name, d in cases_aki.items():
        print(name)
        print(
            f"   P_Na={d['P_Na']}, U_Na={d['U_Na']} mEq/L ; P_cr={d['P_cr']}, U_cr={d['U_cr']} mg/dL ; "
            f"BUN={d['P_bun']}, U_urea N={d['U_urea']} mg/dL"
        )
        if reveal:
            print(
                f"   -> FE_Na = {fe(d['U_Na'], d['P_Na'], d['U_cr'], d['P_cr']):.2f} %, "
                f"FE_urea = {fe(d['U_urea'], d['P_bun'], d['U_cr'], d['P_cr']):.0f} %"
            )


def glucose_titration(GFR=125, Tm=375, theta=0.99):
    Pg = np.linspace(0, 800, 801)  # plasma glucose, mg/dL
    F = GFR * Pg / 100  # filtered load, mg/min
    # non-rectangular hyperbola: smooth min(F, Tm); theta -> 1 gives a sharp corner
    th = min(theta, 0.9999)
    R = ((F + Tm) - np.sqrt((F + Tm) ** 2 - 4 * th * F * Tm)) / (2 * th)
    E = F - R
    thr = (
        Pg[np.argmax(E > 5)] if np.any(E > 5) else np.nan
    )  # glucosuria > 5 mg/min taken as 'detectable'
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(Pg, F, label="Filtered")
    ax.plot(Pg, R, label="Reabsorbed")
    ax.plot(Pg, E, label="Excreted")
    ax.axhline(Tm, ls=":", c="gray")
    ax.axvline(thr, ls="--", c="k", lw=0.8)
    ax.set(
        xlabel="Plasma glucose (mg/dL)",
        ylabel="mg/min",
        ylim=(0, 900),
        title=f"Renal threshold ~ {thr:.0f} mg/dL ; ideal (Tm/GFR) = {100*Tm/GFR:.0f} mg/dL",
    )
    ax.legend()
    plt.show()


def landis(C):
    """Plasma oncotic pressure (mmHg) from protein concentration C (g/dL), Landis-Pappenheimer."""
    return 2.1 * C + 0.16 * C**2 + 0.009 * C**3


def glomerulus(Ra=1.0, Re=1.0, MAP=100, Kf=8.0, C0=6.0, P_BS=15, Hct=0.45, n=2000):
    """Ra, Re: multipliers of baseline afferent/efferent resistance. Returns profiles and summary."""
    ra, re = (
        0.45 * Ra,
        0.55 * Re,
    )  # baseline: total R = 1 (100 mmHg -> 1100 mL/min blood)
    RBF = 1100 * (MAP / 100) / (ra + re)
    RPF = RBF * (1 - Hct)
    P_GC = MAP - RBF * ra * 100 / 1100
    x = np.linspace(0, 1, n + 1)
    Q = np.empty(n + 1)
    Q[0] = RPF
    for i in range(n):
        net = P_GC - P_BS - landis(C0 * RPF / Q[i])
        Q[i + 1] = Q[i] - Kf * max(net, 0) / n
    pi = landis(C0 * RPF / Q)
    GFR = RPF - Q[-1]
    eq = (
        np.argmax(P_GC - P_BS - pi <= 1e-6)
        if np.any(P_GC - P_BS - pi <= 1e-6)
        else None
    )
    return dict(
        x=x,
        pi=pi,
        P_GC=P_GC,
        P_BS=P_BS,
        RPF=RPF,
        GFR=GFR,
        FF=GFR / RPF,
        eq=None if eq is None else x[eq],
    )


def plot_glomerulus(Ra=1.0, Re=1.0, MAP=100, Kf=8.0, C0=6.0, P_BS=15):
    r = glomerulus(Ra, Re, MAP, Kf, C0, P_BS)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    dP = r["P_GC"] - r["P_BS"]
    ax.plot(r["x"], np.full_like(r["x"], dP), label="ΔP = P_GC - P_BS")
    ax.plot(r["x"], r["pi"], label="π_GC")
    ax.fill_between(
        r["x"], r["pi"], dP, where=dP > r["pi"], alpha=0.2, label="net filtration"
    )
    if r["eq"] is not None:
        ax.axvline(r["eq"], ls="--", c="k", lw=0.8)
    eq_txt = (
        f"equilibrium at x={r['eq']:.2f}" if r["eq"] is not None else "no equilibrium"
    )
    ax.set(
        xlabel="Fractional capillary length",
        ylabel="mmHg",
        ylim=(0, 90),
        title=f"GFR={r['GFR']:.0f}, RPF={r['RPF']:.0f} mL/min, FF={r['FF']:.2f}, P_GC={r['P_GC']:.0f}; {eq_txt}",
    )
    ax.legend(loc="lower right")
    plt.show()


def sweep_resistance():
    m = np.linspace(0.5, 4, 36)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
    for key, lab in [("Ra", "Afferent"), ("Re", "Efferent")]:
        res = [glomerulus(**{key: v}) for v in m]
        axs[0].plot(m, [r["GFR"] for r in res], label=lab)
        axs[1].plot(m, [r["RPF"] for r in res], label=lab)
    axs[0].set(xlabel="Resistance multiplier", ylabel="GFR (mL/min)")
    axs[1].set(xlabel="Resistance multiplier", ylabel="RPF (mL/min)")
    for a in axs:
        a.legend()
    plt.tight_layout()
    plt.show()


def multiplier(n=8, single_effect=200, cycles=200, washout=0.02, inflow=300):
    """Stepwise countercurrent multiplier. index 0 = cortex, n-1 = papilla.
    Returns interstitium history (cycles x n) and final ascending-limb profile."""
    desc = np.full(n, inflow, float)
    asc = desc.copy()
    hist = np.empty((cycles, n))
    for k in range(cycles):
        total = desc + asc  # single effect (solute conserved per level)
        desc, asc = (total + single_effect) / 2, (total - single_effect) / 2
        desc -= washout * (desc - inflow)  # vasa recta removal (toy)
        asc -= washout * (asc - inflow)
        hist[k] = desc  # interstitium equilibrated with descending limb
        bottom = desc[-1]  # fluid shift around the hairpin
        desc = np.r_[inflow, desc[:-1]]
        asc = np.r_[asc[1:], bottom]
    return hist, asc


def show_first_cycles(n=6, k=3):
    """Print the textbook-style table for the first k cycles."""
    for c in range(1, k + 1):
        hist, asc = multiplier(n=n, cycles=c, washout=0.0)
        print(
            f"cycle {c}: interstitium (cortex->papilla) = {np.round(hist[-1]).astype(int)}"
        )


def plot_multiplier(n=8, single_effect=200, cycles=200, washout=0.02):
    hist, asc = multiplier(n, single_effect, cycles, washout)
    fig, axs = plt.subplots(1, 2, figsize=(10.5, 3.8))
    depth = np.arange(n)
    axs[0].plot(depth, hist[-1], "o-", label="Interstitium / descending")
    axs[0].plot(depth, asc, "s-", label="Ascending (after shift)")
    axs[0].axhline(300, c="gray", ls=":")
    axs[0].set(
        xlabel="Depth (0 = cortex)",
        ylabel="mOsm/L",
        ylim=(0, 2000),
        title=f"Papilla {hist[-1, -1]:.0f}; leaving TAL {asc[0]:.0f} mOsm/L",
    )
    axs[0].legend()
    im = axs[1].imshow(
        hist.T, aspect="auto", origin="upper", cmap="viridis", vmin=300, vmax=1500
    )
    axs[1].set(
        xlabel="Cycle", ylabel="Depth (0 = cortex)", title="Build-up of the gradient"
    )
    plt.colorbar(im, ax=axs[1], label="mOsm/L")
    plt.tight_layout()
    plt.show()


def urine_balance(
    osm_load=600, U_min=60, U_max=1200, U_actual=600, intake_L=2.0, P_osm=290
):
    U = np.linspace(40, 1300, 500)
    V = osm_load / U
    V_act = osm_load / U_actual
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(U, V, c="k")
    ax.axvspan(U_min, U_max, alpha=0.15, label="Achievable U_osm")
    ax.plot(U_actual, V_act, "ro", label="Current state")
    ax.axhline(intake_L, ls="--", c="tab:blue", label="Water intake")
    ax.set(
        xlabel="Urine osmolality (mOsm/kg)",
        ylabel="Urine volume (L/day)",
        ylim=(0, 12),
        title=f"V range: {osm_load/U_max:.2f}-{osm_load/U_min:.2f} L/day",
    )
    ax.legend()
    plt.show()
    C_osm = osm_load / P_osm
    print(
        f"Current V = {V_act:.2f} L/day ; C_osm = {C_osm:.2f} L/day ; C_H2O = {V_act - C_osm:+.2f} L/day"
    )
    print(
        f"Max excretable water at this load = {osm_load/U_min:.2f} L/day ; "
        f"intake - max excretion = {intake_L - osm_load/U_min:+.2f} L/day"
    )


cases_ab = {
    "1. 설사 5일, 20세 여": dict(
        pH=7.29, PaCO2=26, HCO3=12, Na=138, Cl=116, K=3.1, Alb=4.0
    ),
    "2. 1형 당뇨, 복통·구토": dict(
        pH=7.26, PaCO2=23, HCO3=10, Na=132, Cl=96, K=5.6, Alb=4.0
    ),
    "3. COPD, 폐렴으로 악화": dict(
        pH=7.25, PaCO2=75, HCO3=32, Na=140, Cl=96, K=4.2, Alb=3.5
    ),
    "4. 반복 구토 1주": dict(pH=7.52, PaCO2=48, HCO3=38, Na=140, Cl=88, K=2.9, Alb=4.2),
    "5. Aspirin 과량 복용, 40세": dict(
        pH=7.40, PaCO2=20, HCO3=12, Na=140, Cl=102, K=3.8, Alb=4.0
    ),
    "6. 패혈증, 간경변 (Alb 2.0)": dict(
        pH=7.34, PaCO2=31, HCO3=16, Na=136, Cl=108, K=4.5, Alb=2.0
    ),
}


def hh_pH(hco3, paco2):
    return 6.1 + np.log10(hco3 / (0.03 * paco2))


def analyze_abg(pH, PaCO2, HCO3, Na, Cl, Alb=4.0, K=None, AG_normal=12):
    out = []
    ph_calc = hh_pH(HCO3, PaCO2)
    out.append(
        f"[1] H-H pH = {ph_calc:.2f} (reported {pH:.2f})"
        + (
            "  ** inconsistent: re-check sample **"
            if abs(ph_calc - pH) > 0.03
            else "  OK"
        )
    )
    status = "acidemia" if pH < 7.35 else "alkalemia" if pH > 7.45 else "normal pH"
    out.append(f"[2] {status}")
    primary = []
    if pH < 7.40:
        if HCO3 < 22:
            primary.append("metabolic acidosis")
        if PaCO2 > 44:
            primary.append("respiratory acidosis")
    else:
        if HCO3 > 26:
            primary.append("metabolic alkalosis")
        if PaCO2 < 36:
            primary.append("respiratory alkalosis")
    if len(primary) == 2:
        out.append(
            f"[3] both components point the same way: mixed {primary[0]} + {primary[1]}"
        )
    elif primary:
        p = primary[0]
        out.append(f"[3] primary: {p}")
        if p == "metabolic acidosis":
            exp = 1.5 * HCO3 + 8
            msg = (
                "appropriate"
                if abs(PaCO2 - exp) <= 2
                else (
                    "+ concurrent respiratory alkalosis"
                    if PaCO2 < exp - 2
                    else "+ concurrent respiratory acidosis"
                )
            )
            out.append(
                f"[4] Winter: expected PaCO2 {exp:.0f}±2, actual {PaCO2} -> {msg}"
            )
        elif p == "metabolic alkalosis":
            exp = 0.7 * HCO3 + 21
            msg = (
                "appropriate"
                if abs(PaCO2 - exp) <= 2
                else (
                    "+ concurrent respiratory alkalosis"
                    if PaCO2 < exp - 2
                    else "+ concurrent respiratory acidosis"
                )
            )
            out.append(f"[4] expected PaCO2 {exp:.0f}±2, actual {PaCO2} -> {msg}")
        elif p == "respiratory acidosis":
            d = PaCO2 - 40
            a, c = 24 + 0.1 * d, 24 + 0.35 * d
            msg = (
                "+ concurrent metabolic acidosis"
                if HCO3 < a - 2
                else (
                    "+ concurrent metabolic alkalosis"
                    if HCO3 > c + 2
                    else (
                        "near acute"
                        if HCO3 <= a + 2
                        else (
                            "near chronic"
                            if HCO3 >= c - 2
                            else "between acute and chronic (acute-on-chronic?)"
                        )
                    )
                )
            )
            out.append(
                f"[4] expected HCO3: acute {a:.1f}, chronic {c:.1f}; actual {HCO3} -> {msg}"
            )
        else:
            d = 40 - PaCO2
            a, c = 24 - 0.2 * d, 24 - 0.4 * d
            msg = (
                "+ concurrent metabolic acidosis"
                if HCO3 < c - 2
                else (
                    "+ concurrent metabolic alkalosis"
                    if HCO3 > a + 2
                    else (
                        "near acute"
                        if HCO3 >= a - 2
                        else (
                            "near chronic"
                            if HCO3 <= c + 2
                            else "between acute and chronic"
                        )
                    )
                )
            )
            out.append(
                f"[4] expected HCO3: acute {a:.1f}, chronic {c:.1f}; actual {HCO3} -> {msg}"
            )
    else:
        out.append("[3] no primary disorder by these cut-offs")
    AG = Na - (Cl + HCO3)
    AGc = AG + 2.5 * (4.0 - Alb)
    out.append(
        f"[5] AG = {AG:.0f}; albumin-corrected AG = {AGc:.1f}"
        + ("  -> high AG" if AGc > AG_normal + 2 else "  -> normal AG")
    )
    if HCO3 < 24 and AGc > AG_normal + 2:
        dd = (AGc - AG_normal) / (24 - HCO3)
        hint = (
            "<1: + NAGMA"
            if dd < 1
            else (
                ">2: + metabolic alkalosis"
                if dd > 2
                else "1-2: AG rise accounts for the HCO3 fall"
            )
        )
        out.append(f"[6] delta ratio = {dd:.2f} ({hint})")
    print("\n".join(out))


def plot_davenport(cases):
    pH = np.linspace(7.0, 7.7, 200)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for pc in [20, 30, 40, 60, 80]:
        ax.plot(pH, 0.03 * pc * 10 ** (pH - 6.1), c="gray", lw=0.8)
        ax.text(7.0, 0.03 * pc * 10 ** (0.9), f"{pc}", fontsize=8, color="gray")
    ax.plot(
        pH, 24 - 28 * (pH - 7.40), "k--", lw=0.8, label="Blood buffer line (approx.)"
    )
    for name, d in cases.items():
        ax.plot(d["pH"], d["HCO3"], "o")
        ax.annotate(
            name.split(".")[0],
            (d["pH"], d["HCO3"]),
            xytext=(4, 4),
            textcoords="offset points",
        )
    ax.plot(7.40, 24, "k*", ms=12, label="Normal")
    ax.set(
        xlabel="pH",
        ylabel="[HCO3-] (mEq/L)",
        xlim=(7.0, 7.7),
        ylim=(0, 50),
        title="Davenport diagram (gray: PaCO2 isobars, mmHg)",
    )
    ax.legend(loc="upper left")
    plt.show()

print("✅ 준비 완료. 이제 아래로 내려가며 실습을 진행하세요.")

## M1. Clearance와 glucose titration (20분)

### M1-1. 한 환자의 정량 평가
건강한 성인 남성의 timed urine collection 자료다.

| 물질 | 혈장 (P) | 소변 (U) |
|---|---|---|
| Inulin | 0.5 mg/mL | 55 mg/mL |
| PAH | 0.02 mg/mL | 10 mg/mL |
| Creatinine | 1.0 mg/dL | 115 mg/dL |
| Na⁺ | 140 mEq/L | 40 mEq/L |

소변 유량 V = 1.2 mL/min, Hct = 0.45

Clearance 공식은 **C = U×V/P** 이다. 아래 ▶ 셀을 실행해 GFR(inulin), RPF(PAH), filtration fraction(FF = GFR/RPF), FE_Na를 확인하고, 각 값이 무엇을 의미하는지 생각해 보자.

In [ ]:
#@title ▶ M1-1 계산 결과 보기
check_m1_1()

🤔 **생각해 볼 질문** (제출하지 않아도 됩니다)
1. Creatinine clearance가 inulin clearance보다 큰 이유는?

### M1-2. Glucose titration curve
Filtered load = GFR × 혈장 glucose 농도. 근위세뇨관의 재흡수에는 최대치(**Tm**)가 있어서, filtered load가 Tm을 넘으면 넘친 만큼 소변으로 나온다(당뇨). 소변에 당이 나오기 시작하는 혈장 농도를 **renal threshold**라 한다.

🤔 **생각해 보기** (실행 전에 예상만 해 보자)
- SGLT2 억제제를 쓰면(Tm ↓) renal threshold는 올라갈까, 내려갈까?
- 임신으로 GFR이 늘면 threshold는 어떻게 될까?

In [ ]:
#@title ▶ M1-3 Glucose titration 슬라이더
interact(
    glucose_titration,
    GFR=IntSlider(value=125, min=40, max=200, step=5),
    Tm=IntSlider(value=375, min=50, max=500, step=25),
    theta=FloatSlider(value=0.99, min=0.90, max=1.0, step=0.005),
)

**확인해 보기**: 슬라이더로 아래 조건을 만들고 그래프 제목의 threshold를 보자. 예상과 같았는가? (theta는 건드리지 않아도 된다.)
- (a) 정상: GFR 125, Tm 375
- (b) SGLT2 억제제: Tm 125
- (c) 임신: GFR 175

## M2. 사구체 Starling 역학 (20분)

사구체 모세혈관을 따라 혈장이 여과되면 단백질이 농축되고, 그만큼 oncotic pressure(π_GC)가 상승한다. 순여과압이 0이 되는 지점을 **filtration equilibrium**이라 한다.

그래프 보는 법: 파란 선(ΔP = P_GC − P_BS)은 혈관 밖으로 미는 압력, 주황 선(π_GC)은 안으로 당기는 압력이다. 두 선 사이 색칠된 면적만큼 여과가 일어난다.

슬라이더: **Ra** = afferent(들세동맥) 저항 배수, **Re** = efferent(날세동맥) 저항 배수, **P_BS** = Bowman 주머니 압력. (MAP, Kf, C0은 기본값으로 둔다.)

🤔 **생각해 보기** (실행 전에 예상만 해 보자): 각 상황에서 GFR과 RPF는 늘어날까, 줄어들까?
- NSAID (afferent 수축) → Ra 1.5
- ACE inhibitor (efferent 확장) → Re 0.7
- 요로폐쇄 → P_BS 30

예상해 본 뒤 아래 슬라이더로 확인하자.

In [ ]:
#@title ▶ M2 사구체 Starling 슬라이더
interact(
    plot_glomerulus,
    Ra=FloatSlider(value=1.0, min=0.5, max=2.5, step=0.05),
    Re=FloatSlider(value=1.0, min=0.5, max=4.0, step=0.05),
    MAP=IntSlider(value=100, min=50, max=180, step=5),
    Kf=FloatSlider(value=8.0, min=2.0, max=20.0, step=0.5),
    C0=FloatSlider(value=6.0, min=3.0, max=9.0, step=0.1),
    P_BS=IntSlider(value=15, min=5, max=40, step=1),
)

🤔 **생각해 볼 질문** (제출하지 않아도 됩니다)
1. 신동맥 협착 환자는 efferent 수축(angiotensin II)으로 GFR을 유지한다. 이 환자에게 ACE inhibitor를 주면 GFR이 급감할 수 있는 이유를 모델로 설명하시오. (Ra를 2.0으로 올린 상태에서 Re를 1.0 → 0.7로 바꿔 보라.)

🤔 **더 생각해 보기**: MAP를 80 → 160으로 올리면 이 모델의 GFR은 어떻게 되는가? 실제 신장과 다른 이유는 무엇인가? (힌트: autoregulation)

## M3. 소변 농축·희석의 정량화 (15분)

하루에 배설해야 하는 용질의 양(osmolar load, 정상 약 600 mOsm/day)은 소변 농도와 무관하게 정해진다. 따라서 다음 관계가 성립한다.

$$V = \frac{\text{osmolar load}}{U_{osm}},\qquad C_{osm} = \frac{\text{osmolar load}}{P_{osm}},\qquad C_{H_2O} = V - C_{osm}$$

- C_H₂O > 0: 용질 없는 물(free water)을 배설하는 상태다.
- C_H₂O < 0: free water를 보존하는 상태다.

슬라이더: **osm_load** = 하루 배설해야 할 용질량, **U_min/U_max** = 콩팥이 만들 수 있는 가장 묽은/진한 소변, **U_actual** = 현재 소변 농도, **intake_L** = 하루 수분 섭취량.

In [ ]:
#@title ▶ M4 소변량 슬라이더
interact(
    urine_balance,
    osm_load=IntSlider(value=600, min=100, max=1500, step=50),
    U_min=IntSlider(value=60, min=40, max=600, step=10),
    U_max=IntSlider(value=1200, min=100, max=1400, step=50),
    U_actual=IntSlider(value=600, min=50, max=1200, step=10),
    intake_L=FloatSlider(value=2.0, min=0.5, max=10.0, step=0.25),
)

**케이스**: 슬라이더로 재현하면서 생각해 보자. (제출하지 않아도 됩니다)

1. **Beer potomania**: 하루 맥주 5 L만 마시고 거의 먹지 않는다 (osm_load 약 200). ADH는 적절히 억제되어 있다 (U_min = U_actual = 60).
   - 하루에 배설할 수 있는 최대 수분량은? 저나트륨혈증이 생기는 이유는?
2. **SIADH**: U_osm이 500으로 고정되어 있다 (U_actual 500), osm_load 600.
   - 수분 섭취를 1.0 L/day로 제한하면 충분한가? 염분이나 urea를 보충해 osm_load를 올리면 왜 도움이 되는가?

## 📤 마무리와 제출 (5분)

PLATO 과제란에 **pptx 파일 1개**를 제출합니다. 파일 이름: `학번_이름_신장실습.pptx`

**슬라이드 1~3: 실행 스크린샷**

각 슬라이드에 그래프 하나씩 넣는다.
1. M1: SGLT2 억제제 조건(Tm 125)의 glucose titration 그래프
2. M2: NSAID, ACE inhibitor, 요로폐쇄 중 하나를 재현한 Starling 그래프
3. M3: Beer potomania 또는 SIADH를 재현한 소변량 그래프

> 📸 스크린샷을 찍기 전에 Colab 오른쪽 위의 **프로필 사진(계정 아이콘)을 클릭**해 **내 이메일 주소가 보이는 창을 띄운 상태**로 찍는다. 그래프와 이메일이 한 화면에 같이 나와야 한다.
> (Windows: `Win+Shift+S`, Mac: `Cmd+Shift+4`)

**슬라이드 4: 감상** (2~5문장)

정답을 쓰는 곳이 아닙니다. 내 생각을 내 말로 적어 주세요. 잘 썼는지보다 직접 해 봤는지를 봅니다.
- 예상과 달라서 놀랐던 결과가 있었다면 무엇이었나요?
- 슬라이더를 움직여 보면서 새로 이해된 것(또는 아직 헷갈리는 것)은?
- 이번 실습 방식에 대한 소감이나 바라는 점이 있다면?